In [ ]:
import numpy as np
import math

import torch
import torch.nn.init as init
import torch.nn.functional as F
# torch.autograd.set_detect_anomaly(True)

import matplotlib.pyplot as plt
from scipy.special import gamma

In [ ]:
seed = np.random.randint(0,200)
seed

In [ ]:
np.random.seed(seed)

In [ ]:
# SMT Solver
# !pip install z3-solver

# from z3 import *

In [ ]:
# dreal/dreal4: Automated Reasoning in Nonlinear Theories of Reals
import pkgutil
if not pkgutil.find_loader("dreal"):
  !curl https://raw.githubusercontent.com/dreal/dreal4/master/setup/ubuntu/22.04/install.sh | bash
  !pip install dreal --upgrade

import dreal as dr

In [ ]:
# LQR Controller
!pip install control

# from control.matlab import *  # MATLAB-like functions
import control as ct

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Fractional Order System Model


In [ ]:
# order of the Caputo fractional derivative
q = 0.7

# initial states X0 = ([x1(t0), x2(t0),.. xN(t0)]), only for plotting original dynamics
initial_X_array = np.array([2, 2])
initial_X_tensor = torch.tensor([2, 2], dtype=torch.float)

# initial starting time
t0 = 0.01

# total number of time steps (= total number of solutions calculated φ(t,x0))
# final time t_{final} = t0 + num_steps*∆t
num_steps = 300

# time step size
Delta_t = 0.01

In [ ]:
# number of state vectors
N = len(initial_X_tensor)

# system constant(s)
a1 = 1        # Prey Growth Rate
a2 = 0.25     # Predation Rate Coefficient
a3 = 0.05     # Predator Death Rate
a4 = 0.8      # Predator Reproduction Rate Coefficient

# the matrix for the linear part of the system dynamics
A = np.array([[a1, 0], [0, -a4]])

# controller acting on the non-linear part
B = np.array([[1], [1]])

In [ ]:
# controlled dynamics for the system (TENSOR)
def f_controlled(t, X, K):
    x, y = X[0], X[1]

    A = torch.tensor([[a1, 0], [0, -a4]], dtype=torch.float)
    B = torch.tensor([[1], [1]], dtype=torch.float)

    # calculate the linear and controller parts (TENSOR)
    Linear = torch.matmul(A, X.unsqueeze(-1)).squeeze(-1)
    Control = torch.matmul(torch.matmul(B, K), X.unsqueeze(-1)).squeeze(-1)

    # non-linear term
    # nonLinear = torch.tensor([-a2*x1*x2, a3*x1*x2])

    # controlled system dynamics
    x_u = Linear[0] - Control[0] - a2*x*y
    y_u = Linear[1] - Control[1] + a3*x*y

    return torch.stack([x_u, y_u])

In [ ]:
# controlled dynamics for the system (ARRAY)
def f_ce_controlled(t, X, K):
    x, y = X[0], X[1]

    Linear = np.dot(A, X)
    Control = np.dot(np.matmul(B, K), X)

    x_u = Linear[0] - Control[0] - a2*x*y
    y_u = Linear[1] - Control[1] + a3*x*y

    return np.array([x_u, y_u])

## Numerical Solver for Computing the State Vectors

In [ ]:
# compute the controlled dynamics
def fu_solver(f_ode, initial_states, K, q, t0, h, num_steps):

    # Create a zero array *sol* of size (num_steps, N)
    N = len(initial_states)
    sol = torch.zeros((num_steps+1, N)) # that's of size (num_steps, N)
    t_interval = [t0]

    # Set initial conditions
    sol[0] = initial_states

    f_t1 = f_ode(t0+h, sol[0], K)
    sol[1] = sol[0] + ((h**q)/gamma(q+1))*f_t1

    # Fractional Adams-Bashforth Method
    for k in range(1, num_steps):
        t = round(t0+k*h, 5)
        t_interval.append(t)

        sum_prev = 0
        if k > 1:
            sum_prev = (sol[k-1] - sol[k-2]) / (h**q)
        f_tk = f_ode(t, sol[k], K)

        sol[k+1] = sol[k] + ((h**q)/gamma(q+1))*(f_tk)

    return sol

### Controller Gain Matrix $K_{lqr}$ and Lyapunov Function $V_{lqr} = \frac{1}{2} \thinspace \textbf{x}^T \thinspace P \thinspace\textbf{x} $ Derived from Solving the ARE

In [ ]:
# define the matrices Q and R
R = np.array([1])
Q = np.eye(N)

# solve for optimal control matrix K_lqr:
# S is the inverse of P, the solution of the Algebraic Riccati Equation (ARE)
# S will be used later for constructing the V_lqr function
(K_lqr, S, E) = ct.lqr(A, B, Q, R)
print("LQR Feedback matrix K_lqr: {}".format(K_lqr))

In [ ]:
# solve the Algebraic Riccati Equation (ARE) and get P
# (K_lqr, S, E) = ct.lqr(A, B, Q, R) solved in a the section above
P_lqr = np.linalg.inv(S)

In [ ]:
print('Solution of the Algebraic Riccati Equation (ARE):')
print(P_lqr)

In [ ]:
# Lyapunov function V_lqr = 1/2 X.t() P X
def V_lqr(in_P, in_x1, in_x2):
    # separate each element of P
    Pa = in_P[0][0]; Pb = in_P[0][1]
    Pc = in_P[1][0]; Pd = in_P[1][1]
    return Pa*(in_x1**2)+(Pb+Pc)*(in_x1*in_x2)+Pd*(in_x2**2)

## Neural Network Model

In [ ]:
class Cosh_Activation(torch.nn.Module):
    def __init__(self):
        super(Cosh_Activation, self).__init__()

    def forward(self, x):
        return torch.cosh(x)-1

In [ ]:
class KVNet(torch.nn.Module):
    # def __init__(self, n_input, n_hidden, n_output, hid_w, hid_b, out_w, out_b, lqr): # manually-defined weights
    def __init__(self, n_input, n_hidden, n_output, lqr):
        super(KVNet, self).__init__()
        torch.manual_seed(2)
        self.layer1 = torch.nn.Linear(n_input, n_hidden)
        # init.xavier_uniform_(self.layer1.weight)
        # self.layer1.weight = torch.nn.Parameter(hid_w)     # initialize the weights
        # self.layer1.bias = torch.nn.Parameter(hid_b)       # initialize the bias terms

        self.layer2 = torch.nn.Linear(n_hidden, n_output)
        # init.xavier_uniform_(self.layer2.weight)
        # self.layer2.weight = torch.nn.Parameter(out_w)
        # self.layer2.bias = torch.nn.Parameter(out_b)

        self.control = torch.nn.Linear(n_input, 1, bias=False)
        self.control.weight = torch.nn.Parameter(lqr)       # initial weight set as lqr

        self.activation = Cosh_Activation()

    def forward(self, X):
        X = X.float()
        h_1 = self.activation(self.layer1(X))
        V = self.activation(self.layer2(h_1))
        # controlled Caputo fractional order dynamics for calculating DV
        # fu = [x + ((h**q)/gamma(q+1))*f_controlled(t0+h, x, self.control.weight) for x in X]
        K_nn = self.control.weight

        return V, K_nn

### NN initialization

In [ ]:
# number of input features
n_input = N

# number of hidden nodes
n_hidden = 8

# number of output features
n_output = 1
torch.manual_seed(10)

# manually defined hidden layer and output layer weights for 8 hidden layer nodes
# hidden_weights = torch.tensor(np.random.uniform(-5,5, size=(n_hidden, n_input)) , dtype=torch.float)
# hidden_bias = torch.tensor([np.random.uniform(-5,5) for _ in range(n_hidden)], dtype=torch.float)
# hidden_weights = torch.tensor([[-0.6, 0.6], [0.5, 0.6], [0.2, 0.1]] , dtype=torch.float)
# hidden_bias = torch.tensor([-0.3, -0.02, 0.03], dtype=torch.float)
# output_weights = torch.tensor([[0.6, 2.8, 0.01]], dtype=torch.float)
# output_bias = torch.tensor([0.01], dtype=torch.float)

# lqr controller gain matrix K_lqr
lqr = torch.tensor(K_lqr, dtype=torch.float, requires_grad=True)

# building the NN
# NN = KVNet(n_input, n_hidden, n_output, hidden_weights, hidden_bias, output_weights, output_bias, lqr)
NN = KVNet(n_input, n_hidden, n_output, lqr)

In [ ]:
# optimizer
learning_rate = 0.02
optimizer = torch.optim.Adam(NN.parameters(), lr=learning_rate)
# optimizer = torch.optim.SGD(NN.parameters(), lr=learning_rate, momentum=0.9, weight_decay=0.1)

In [ ]:
for name, param in NN.named_parameters():
    if param.requires_grad:
        print(f"{name}: {param.size()}")

## Functions Needed

In [ ]:
# the class-K function alpha()
def alpha(X, sample_index):
    x1, x2 = X[:,0][sample_index], X[:,1][sample_index]
    norm = torch.sqrt((torch.pow(x1,2) + torch.pow(x2,2)))
    return 0.3*torch.pow(norm,(3/2))

# the class-K function alpha()
def alpha_calc(X):
    x1, x2 = X[0], X[1]
    norm = (x1**2 + x2**2)**(1/2)
    return 0.3*((norm)**(3/2))

In [ ]:
# math cosh function
def cosh_calc(X):
    numerator = math.e**(X) + math.e**(-X)
    denominator = 2
    return numerator/denominator

# math sinh function
def sinh_calc(X):
    numerator = math.e**(X) - math.e**(-X)
    denominator = 2
    return numerator/denominator

# math cosh-1 function
def cosh_minus1_calc(X):
    numerator = math.e**(X) + math.e**(-X)
    denominator = 2
    return (numerator/denominator) - 1

In [ ]:
# maximum number of iterations
max_iters = 500

# Lyapunov function candidate V's validity
V_valid = False

# initialization
Losses = []           # list of losses
itera = 0             # iteration looping constant
constant_c = 0.1      # positive constant, introduced in Equation (9): D V ≤ −c V

var_epsilon = 0.035   # tolerable numerical error for V(0)
epsilon_0 = 0.125     # closed ϵ0-neighborhood cal_N
eta = 1.75            # radius of B(η), within the original dynamics's I
num_samples = 50      # initial number of inputs
PREV_CE = [0, 0]      # initialize the list storing the counterexample generated

N = len(initial_X_tensor)

# initial input values into the NN
# random x0 inside the η-ball B(η), the code below will not make xᵢ reach the exact value of -η or η
X = torch.Tensor(num_samples, N).uniform_(-eta, eta)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from mpl_toolkits.mplot3d import axes3d

%matplotlib inline
%config InlineBackend.figure_format = 'svg'

In [ ]:
def plotV(beta, omega, w, b):

    PLOT_x1 = np.linspace(-3, 3, 200)
    PLOT_x2 = np.linspace(-3, 3, 200)

    x1, x2 = np.meshgrid(PLOT_x1, PLOT_x2)

    # the Lyapunov Function above
    def V(in_x1, in_x2):
        sumZ = 0
        for j in range(n_hidden):
            sumZ = sumZ + omega[0][j]*(np.cosh(w[j][0]*in_x1+w[j][1]*in_x2+b[j])-1)
        return np.cosh(sumZ+beta)-1

    # Lyapunov function V_nn (clipped for plotting 3d surface)
    V_clipped = np.clip(V(x1,x2), -5, 10)
    V_clipped[V_clipped == 10] = np.nan

    # ++++++++++++++++++++++++++++++++++++++++++++++++++++++

    fig = plt.figure(figsize=(6,8))
    ax = fig.add_subplot(111, projection='3d')
    # number of values corresponding to the isocontour closest to r=eta
    num_levels = 5

    # the Lyapunov scalar field V_nn
    V_nn_surf = ax.plot_surface(x1, x2, V_clipped, rstride=5, cstride=5, alpha=0.55, cmap=plt.cm.viridis)
    ax.set_zlim(0, 10)
    fig.colorbar(V_nn_surf, ax=ax, shrink=0.3, aspect=5, pad=0.15)

    # class-Κ Function α, appears in Equation (8)
    alpha_surf = ax.plot_surface(x1, x2, alpha_calc([x1,x2]), rstride=5, cstride=5, alpha=0.6, cmap=plt.cm.magma)
    ax.set_zlim(0, 10)
    # fig.colorbar(alpha_surf, ax=ax, shrink=0.4, aspect=5, pad=0.2)

    # isocontour closest to the radius r=eta
    contour_NN = ax.contour(x1, x2, V_clipped, [3, 6, 9, 12, 15], zdir='z', offset=0, cmap=plt.cm.viridis)

    plt.clabel(contour_NN, fontsize=10, colors='black')

    # valid region
    theta = np.linspace(0, 2*np.pi, 50)
    Valid_x1 = eta*np.cos(theta)
    Valid_x2 = eta*np.sin(theta)
    ax.plot(Valid_x1[:], Valid_x2[:], 'black', linestyle='--', linewidth=2, label='B(η)')
    plt.legend()

    ax.set_xlabel('x1'); ax.set_ylabel('x2'); ax.set_zlabel('V*')

    plt.show()

# Main Function with SMT Solver to Get $K_{NN}$ and $V_{NN}$


In [ ]:
torch.autograd.set_detect_anomaly(True)

In [ ]:
# update NN weights
while itera < max_iters and V_valid == False:

    # V_candidate is the numerical result after nested nn.Tanh(), size = (num_samples, N)
    V_candidate, K_nn = NN(X)

    # check the zero input for V(0) = 0
    zero_input = torch.zeros((1, N))
    V_zero, K_nn_zero = NN(zero_input)

    # ++++++++++++++++++++++++++++++++++++++++++++++++++++ model parameters

    w = NN.layer1.weight.data.numpy()
    omega = NN.layer2.weight.data.numpy()
    b = NN.layer1.bias.data.numpy()
    beta = NN.layer2.bias.data.numpy()
    Knn = NN.control.weight.data.numpy()

    # ++++++++++++++++++++++++++++++++++++++++++++++++++++

    # derivative of Lyapunov function V
    # D_V = Caputo_frac_derivative_calculator(round(h*num_steps, 5), h, V_candidate, q)
    # CaputoD_V = caputo(f=v, alpha=q, lower=epsilon_0, upper=eta, quadrature='rs', n=200)
    # D_V = CaputoD_V['fd']

    # D_V = sum of partial derivatives ∑∂V/∂xᵢ*fᵢ = (dV/dζ)*(∂ζ/∂xᵢ)*(dxᵢ/dt)

    # ===================================================
    # calculate model loss, here relu() is used as max(0,·)
    loss_components = []
    sum_V_condi = 0
    sum_DV_condi = 0

    # Calculate alpha, V, DV, V(0)
    for sam in range(0, len(X)):
        # Calculate DV, (approxi) dV/dt = (dV/dζ)*Σ((∂ζ/∂xᵢ)*(dxᵢ/dt))

        # dV/dζ
        # ζ = β + (ω1*(cosh(z1)-1)+ω2*(cosh(z2)-1)+ω3*(cosh(z3)-1))+ω4*(cosh(z4)-1))
        zeta = NN.layer2.bias + torch.matmul(NN.layer2.weight, torch.cosh(NN.layer1.bias + torch.matmul(X[sam], NN.layer1.weight.T))-1)
        dVdZ = torch.sinh(zeta)

        # ∂ζ/∂X
        # ∂ζ/∂xᵢ = Σ(ω1*w1ᵢ*sinh(z1) + ω2*w2ᵢ*sinh(z2) + ω3*w3ᵢ*sinh(z3) + ...)
        # z = b + (w1*x1+w2*x2)
        z = NN.layer1.bias + torch.matmul(X[sam], (NN.layer1.weight.T))

        pZpX = []
        for i in range(n_input):
            pZpx = 0
            for j in range(n_hidden):
                pZpx = pZpx + NN.layer2.weight[0][j]*NN.layer1.weight[j][i]*torch.sinh(z[j])
            pZpX.append(pZpx)  # pZpX = [∂ζ/∂x1, ∂ζ/∂x2]

        # dX/dt
        # dxᵢ/dt = state variables of the controlled dynamics
        # dXdt = ((h**q)/gamma(q+1))*f_controlled(t0, X[sam], NN.control.weight)
        dXdt = f_controlled(t0, X[sam], NN.control.weight)

        # for nts in range(1,num_time_step):
        #     prev = fu_X
        #     prev_augment = ((h**q)/gamma(q+1))*f_controlled(t0+nts*h, prev, NN.control.weight)
        #     fu_X = prev + prev_augment

        # dXdt = fu_X  # dXdt = [dx1/dt, dx2/dt] (n_input=2)

        # Derivative of V (approxi) = (dV/dζ)*Σ((∂ζ/∂xᵢ)*(dxᵢ/dt))
        dZdt = 0
        for i in range(n_input):
            dZdt = dZdt + pZpX[i]*dXdt[i]
        D_V = dVdZ * dZdt

        V_condi = F.relu(alpha(X, sam)-V_candidate[sam])
        DV_condi_1 = F.relu(D_V + constant_c*V_candidate[sam])
        DV_condi_2 = F.relu(torch.tensor(-500)-D_V) # gradient should be Lipschitz continuous; see (5)

        sum_V_condi = sum_V_condi + V_condi   # saved for checking the condition
        sum_DV_condi = sum_DV_condi + DV_condi_1 + DV_condi_2

        loss_components.append(V_condi + DV_condi_1 + DV_condi_2 + V_zero**2) # - torch.min(D_V, -constant_c*V_candidate[sam]-5)) # not too negative?
        # + dis(fu_X[sam]) # (K_nn updating)

    Loss = sum(loss_components) # / len(X)


    # append loss to the list of losses
    Losses.append(Loss.item())

    print('')
    print('Iteration', itera, 'with model loss:', round(Loss.item(),7))


    # at t_final = t_0 + num_steps*∆t,
    # can V_candidate(t,x(t)) < epsilon_0?
    V_t_final_Valid = True

    # current V_candidate
    def V_candidate(in_x1, in_x2):
        sumZ = 0
        for j in range(n_hidden):
            sumZ = sumZ + omega[0][j]*(np.cosh(w[j][0]*in_x1+w[j][1]*in_x2+b[j])-1)
        return np.cosh(sumZ+beta)-1

    max_V_t_final = 0

    for sam in range(0, len(X)):
        f_controlled_candidate = fu_solver(f_controlled, X[sam], torch.tensor(Knn, dtype=torch.float), q, t0, Delta_t, num_steps)
        candidate_x1, candidate_x2 = f_controlled_candidate[:,0], f_controlled_candidate[:,1]

        V_t_final = V_candidate(candidate_x1[-1], candidate_x2[-1])

        if V_t_final.item() > max_V_t_final:
            max_V_t_final = V_t_final.item()

    if max_V_t_final > epsilon_0:
        V_t_final_Valid = False

    if itera%10 == 0:
        # Show The Loss over Iterations

        # plt.rcParams['font.family'] = 'Times New Roman'
        # indices = [i for i in range(len(Losses))]
        # plt.plot(indices, Losses, label='Loss over iterations', color='teal')
        # plt.legend()
        # plt.grid()
        # plt.show()
        print('')
        print("See NN's Learning Outcomes for Every 10 iterations --------------")
        print('Loss(Alpha <= V):', round(sum_V_condi.item(),7))
        print('Loss(DV <= -cV):', round(sum_DV_condi.item(),7))
        print('Loss(V(0)):', round(V_zero.item(),7))
        print('V(t_final)_Valid:', V_t_final_Valid)
        print('-----------------------------------------------------------------')

        plotV(beta, omega, w, b)



    # ==========================================================================
    # ==========================================================================
    # ==========================================================================


    if (sum_V_condi.item() == 0) and (sum_DV_condi.item() == 0) and (Loss.item()/len(X) <= var_epsilon) and V_t_final_Valid: #  = (sum_V_zero.item() <= var_epsilon)
        V_valid = True # ending the main loop
        print('Conditions Satisfied')
        print('Program Ends')
        print('Current Gain Matrix K_nn:', Knn)
        print('Current Max V(t_final, x(t_final)):', max_V_t_final)


    else:
        print('Conditions NOT Satisfied')

        optimizer.zero_grad(set_to_none=True)

        # backward propagation for updating model parameters
        Loss.backward()
        optimizer.step()

        # SMT Falsification

        # if the constraints are not satisfied at any of the sample point, iteration_result will be False
        iteration_result = True

        # SMT Solver
        ce_x1 = dr.Variable('ce_x1')
        ce_x2 = dr.Variable('ce_x2')

        # calculate alpha, V, DV
        CE_alpha = alpha_calc([ce_x1, ce_x2])  # <<++++++++++++++++++++ CE_alpha

        out_1 = np.dot([ce_x1, ce_x2], w.T) + b
        out_M = []

        for j in range(len(out_1)):
            out_M.append(dr.cosh(out_1[j]))

        out_2 = 0
        for k in range(len(out_M)):
            omegaVal = omega.T[k].item()
            out_2 = out_2 + (out_M[k] * omegaVal)

        # if V_zero.item() >= 0:
            # CE_V = dr.cosh(out_2 + beta.item())-V_zero.item()
        # elif V_zero.item  < 0:
            # CE_V = dr.cosh(out_2 + beta.item())+V_zero.item()

        CE_V = dr.cosh(out_2 + beta.item())  # <<++++++++++++++++++++++++++ CE_V

        # zᵢ = bᵢ + (w1ᵢ*x1+w2ᵢ*x2)
        CE_z = b + np.dot([ce_x1, ce_x2], w.T)

        # ζ = β+(ω1*(cosh(z1)-1) + ω2*(cosh(z2)-1) + ω3*(cosh(z3)-1)) + ω4*...
        CE_exp_z = [dr.exp(zi) for zi in CE_z]
        CE_exp_negz = [dr.exp(-zi) for zi in CE_z]
        CE_cosh_z_minus1 = []
        for i in range(len(CE_exp_z)):
            CE_cosh_z_minus1.append(((CE_exp_z[i]+CE_exp_negz[i])/2)-1)
        CE_zeta = beta.item() + (np.dot(omega, CE_cosh_z_minus1)).item()

        # dV/dζ = sinh(ζ)
        CE_dVdZ = (dr.exp(CE_zeta) - dr.exp(-CE_zeta))/2

        # ∂ζ/∂X
        CE_pZpX = []
        for i in range(n_input):
            CE_pZpx = 0
            for j in range(n_hidden):
                CE_pZpx = omega[0][j]*w[j][i]*dr.sinh(CE_z[j])
            CE_pZpX.append(CE_pZpx)

        # dX/dt
        # dxᵢ/dt = fᵢ = state variables of the controlled dynamics
        # CE_dXdt = ((h**q)/gamma(q+1))*f_ce_controlled(t0, [ce_x1, ce_x2], Knn)
        CE_dXdt = f_ce_controlled(t0, [ce_x1, ce_x2], Knn)

        # dV/∂t = (dV/dζ)*(∂ζ/∂xᵢ)*(dxᵢ/dt)
        # dV/∂t = sinh(ζ)*Σ(ωᵢ*d/dxᵢ(cosh(w1ᵢ*x1+w2ᵢ*x2)))
        CE_dZdt = 0
        for i in range(n_input):
            CE_dZdt = CE_dZdt + CE_pZpX[i]*CE_dXdt[i]
        CE_DV = CE_dVdZ * CE_dZdt  # <<+++++++++++++++++++++++++++++++++++ CE_DV

        # ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

        # ball bounds
        low = epsilon_0**2
        high = eta**2

        # Counterexample: X=[ce_x1,ce_x2] in ball(), and Satisfiy ¬(Phi Constraints)
        Ball_Constraints = dr.logical_and(0<ce_x1**2, ce_x1**2<high,
                                          0<ce_x2**2, ce_x2**2<high)

        # Phi_Constraints = dr.logical_or(CE_alpha > CE_V, CE_DV > (-constant_c)*CE_V)

        CE_Constraints = dr.logical_and(dr.logical_imply(Ball_Constraints, CE_alpha<=CE_V),
                                        dr.logical_imply(Ball_Constraints, CE_DV<=(-constant_c)*CE_V))

        # CE_result = dr.CheckSatisfiability(dr.logical_and(Ball_Constraints, Phi_Constraints), 0.001)
        CE_result = dr.CheckSatisfiability(dr.logical_not(CE_Constraints), 0.001)

        # if the solver can find a unique CE
        if CE_result:
            CE = [round(CE_result[0].mid(),7), round(CE_result[1].mid(),7)]

            # avoid adding the same counterexample into X
            if CE[0] != PREV_CE[0] and CE[1] != PREV_CE[1]:
                PREV_CE = CE
                print(f"New Counterexample Found: x1 = {CE[0]}, x2 = {CE[1]}")
                # print(f"Counterexample found: c = {new_c_val}, var_epsilon = {new_var_epsilon_val}")
                X_list = []
                for x in X:
                    X_list.append(x)

                X_list.append(torch.tensor(CE, dtype=torch.float))
                X = torch.stack(X_list, dim=0)

            else:
                # just the same
                PREV_CE = CE
                print(f"Counterexample Found: x1 = {CE[0]}, x2 = {CE[1]}")

        else:
            # print('NO Counterexample Found, Your Program Sucks Huh')
            print('NO Counterexample Found')

    itera = itera + 1

# Valid Lyapunov Function $V^{\ast}_{nn}$

In [ ]:
# print V_nn out in a readable form
# state variables
x1 = dr.Variable('x1')
x2 = dr.Variable('x2')

# candidate Lyapunov Function in function form, see Equation (10)
output_1 = np.dot([x1, x2], w.T) + b
output_N = []
for i in range(0, len(output_1)):
    output_N.append(dr.cosh(output_1[i])-1)

output_2 = np.dot(output_N, omega.T) + beta

V_func = dr.cosh(output_2.item(0))-1
print('Final Valid Lyapunov Function:')
print(V_func)

In [ ]:
# the Lyapunov Function V_nn
def V_nn(in_x1, in_x2):
    sumZ = 0
    for j in range(n_hidden):
        sumZ = sumZ + omega[0][j]*(np.cosh(w[j][0]*x1+w[j][1]*x2+b[j])-1)
    return np.cosh(sumZ+beta)-1

In [ ]:
# state variables
x = dr.Variable('x')
y = dr.Variable('y')

# candidate Lyapunov Function in function form, see Equation (10)
output_1 = np.dot([x, y], w.T) + b
output_N = []
for i in range(0, len(output_1)):
    output_N.append(dr.cosh(output_1[i])-1)

output_2 = np.dot(output_N, omega.T) + beta
V_func = dr.cosh(output_2.item(0))-1

print('Final Valid Lyapunov Function:', V_func)

# Plot the Results ($V_{nn}$, Model Loss, ROA)

In [ ]:
# the Lyapunov Function above
def V_nn(in_x1, in_x2):
    sumZ = 0
    for i in range(n_input):
        for j in range(n_hidden):
            sumZ = sumZ + omega[0][j]*(np.cosh(w[j][0]*in_x1+w[j][1]*in_x2+b[j])-1)
    return np.cosh(sumZ+beta)-1

In [ ]:
# plot the 3d surface
PLOT_r = eta + 1.5
PLOT_x1 = np.linspace(-PLOT_r, PLOT_r, 800)
PLOT_x2 = np.linspace(-PLOT_r, PLOT_r, 800)

# two-dimensional grid (x1,x2)
x1, x2 = np.meshgrid(PLOT_x1, PLOT_x2)

In [ ]:
# Lyapunov function V_nn (clipped for plotting 3d surface)
V_nn_clipped = np.clip(V_nn(x1,x2), -5, 35)
V_nn_clipped[V_nn_clipped == 35] = np.nan

# ++++++++++++++++++++++++++++++++++++++++++++++++++++++

fig = plt.figure(figsize=(6,8))
ax = fig.add_subplot(111, projection='3d')
# number of values corresponding to the isocontour closest to r=eta
num_levels = 5

# the Lyapunov scalar field V_nn
V_nn_surf = ax.plot_surface(x1, x2, V_nn_clipped, rstride=5, cstride=5, alpha=0.55, cmap=plt.cm.viridis)
ax.set_zlim(0, 35)
fig.colorbar(V_nn_surf, ax=ax, shrink=0.3, aspect=5, pad=0.15)

# class-Κ Function α, appears in Equation (8)
alpha_surf = ax.plot_surface(x1, x2, alpha_calc([x1,x2]), rstride=5, cstride=5, alpha=0.6, cmap=plt.cm.magma)
ax.set_zlim(0, 35)
# fig.colorbar(alpha_surf, ax=ax, shrink=0.4, aspect=5, pad=0.2)

# isocontour closest to the radius r=eta
contour_NN = ax.contour(x1, x2, V_nn_clipped, [0.2, 0.6, 2, 6, 20, 30], zdir='z', offset=0, cmap=plt.cm.viridis)

plt.clabel(contour_NN, fontsize=10, colors='black')

# valid region
theta = np.linspace(0, 2*np.pi, 200)
Valid_x1 = eta*np.cos(theta)
Valid_x2 = eta*np.sin(theta)
ax.plot(Valid_x1[:], Valid_x2[:], 'black', linestyle='--', linewidth=2, label='B(η)')
plt.legend()

ax.set_xlabel('x1'); ax.set_ylabel('x2'); ax.set_zlabel('V*')
plt.title('        Valid Lyapunov Function V* Computed by NN \n\n         (fractional-order Lotka-Volterra System) \n', fontsize=12)

# legend for the 3d plot
# plt.legend([plt.Rectangle((0,0), 1, 2, color=(80/255, 175/255, 150/255, 0.8), fill=True, linewidth=2),
            # plt.Rectangle((0,0), 1, 2, color=(220/255, 80/255, 100/255, 0.8), fill=True, linewidth=2)],
            # ['Valid Lyapunov Function V*(X)','Class-Κ Function α(||X||)'], bbox_to_anchor=(0.9, -0.1), loc=0, borderaxespad=0, ncol=1)

plt.savefig('V_LKS.png', dpi=1200, bbox_inches='tight')

plt.show()

In [ ]:
# Lyapunov function V_lqr (clipped for plotting 3d surface)
V_lqr_clipped = np.clip(V_lqr(P_lqr,x1,x2), -5, 20)
V_lqr_clipped[V_lqr_clipped == 20] = np.nan

# ++++++++++++++++++++++++++++++++++++++++++++++++++++++

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
# number of values corresponding to the isocontour closest to r=eta
num_levels = 5

# the Lyapunov scalar field V_nn
V_lqr_surf = ax.plot_surface(x1, x2, V_lqr_clipped, rstride=5, cstride=5, alpha=0.5, cmap=plt.cm.viridis)
ax.set_zlim(0, 20)
fig.colorbar(V_lqr_surf, ax=ax, shrink=0.5, aspect=5, pad=0.15)

# class-Κ Function α, appears in Equation (8)
alpha_surf = ax.plot_surface(x1, x2, alpha_calc([x1,x2]), rstride=5, cstride=5, alpha=0.5, cmap=plt.cm.magma)
ax.set_zlim(0, 20)
# fig.colorbar(alpha_surf, ax=ax, shrink=0.4, aspect=5, pad=0.2)

# isocontour closest to the radius r=eta
contour_lqr = ax.contour(x1, x2, V_lqr_clipped, num_levels, zdir='z', offset=0, cmap=plt.cm.viridis)

plt.clabel(contour_lqr, fontsize=10, colors='black')

# valid region
theta = np.linspace(0, 2*np.pi, 50)
Valid_x1 = eta*np.cos(theta)
Valid_x2 = eta*np.sin(theta)
ax.plot(Valid_x1[:], Valid_x2[:], 'g', linestyle='--', linewidth=2, label='Valid region')

ax.set_xlabel('x1'); ax.set_ylabel('x2'); ax.set_zlabel('V')
plt.title('Lyapunov Function V_lqr')

# legend for the 3d plot
plt.legend([plt.Rectangle((0,0), 1, 2, color=(75/255, 225/255, 140/255, 0.6), fill=True, linewidth=2),
            plt.Rectangle((0,0), 1, 2, color=(220/255, 50/255, 80/255, 0.5), fill=True, linewidth=2)],
            ['Lyapunov Function V_lqr','Class-Κ Function α(|X|)'], bbox_to_anchor=(0.9, -0.1), loc=0, borderaxespad=0, ncol=1)

plt.savefig('Final_Lyapunov.png', dpi=1200, bbox_inches='tight')
plt.legend()
plt.show()

### Plot the Model Loss

In [ ]:
# Show The Loss over Iterations

# plt.rcParams['font.family'] = 'Times New Roman'
indices = [i for i in range(len(Losses))]
plt.plot(indices, Losses, label='Model Loss', color='mediumblue')

plt.title('Model Loss over Iterations')
plt.xlabel('Index of Iteration')
plt.ylabel('Model Loss')

plt.legend()
plt.grid(True)
plt.savefig('Model_Loss.png', dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
# plot the direction field of the controlled dynamics
def PlotDF(in_x1, in_x2, t, K, in_color):
    # phase plane of the actual dynamics, use the np version of controlled dynamics
    DF_x1 = []
    DF_x2 = []
    for i in range(len(in_x1)):
        df_x1, df_x2 = f_ce_controlled(t, [in_x1[i], in_x2[i]], K)
        DF_x1.append(df_x1)
        DF_x2.append(df_x2)
    plt.streamplot(in_x1, in_x2, np.array(DF_x1), np.array(DF_x2), color=in_color, linewidth=0.6, density=0.6, arrowstyle='-|>', arrowsize=1)

### Plot the ROA

In [ ]:
r = eta+0.5

ROA_x1 = np.linspace(-r, r, 800)
ROA_x2 = np.linspace(-r, r, 800)
ROA_t = np.linspace(t0, t0+(num_steps*Delta_t), num_steps)

# two-dimensional grid (roa_x1, roa_x2)
roa_x1, roa_x2 = np.meshgrid(ROA_x1, ROA_x2)

plt.figure(figsize=(8.75, 5.25))
ax = plt.gca()

# plot the direction field
PlotDF(roa_x1, roa_x2, ROA_t, Knn, 'darkslateblue')

# valid region
theta = np.linspace(0, 2*np.pi, 200)
Valid_x1 = eta*np.cos(theta)
Valid_x2 = eta*np.sin(theta)
ax.plot(Valid_x1[:], Valid_x2[:], 'black', linestyle='--', linewidth=2, label='B(η)')

# plot the ROA for V_nn
cset_alpha = ax.contourf(roa_x1, roa_x2, alpha_calc([roa_x1, roa_x2]), 10, alpha=0.55, cmap=plt.cm.magma)
# color set, number of color levels = 10
roa_V_nn_clipped = np.clip(V_nn(roa_x1,roa_x2), -5, 2)
roa_V_nn_clipped[roa_V_nn_clipped == 2] = np.nan
cset_NN = ax.contourf(roa_x1, roa_x2, roa_V_nn_clipped, 10, alpha=0.45, cmap=plt.cm.viridis)
plt.colorbar(cset_NN)
plt.colorbar(cset_alpha)

# contour at height [0.1, 0.2, 0.3, 0.4, 0.5]
contour_NN = ax.contour(roa_x1, roa_x2, roa_V_nn_clipped, [0.2, 0.3, 0.6, 0.9, 1.5, 1.8], linewidths=1.5, cmap=plt.cm.viridis)
# contour_lqr = ax.contour(roa_x1, roa_x2, V_lqr(P_lqr,roa_x1,roa_x2), [0.2], linewidths=1.5, colors='mediumblue', linestyles='-')

# specify the heights
plt.clabel(contour_NN, fontsize=10, colors='black')
# plt.clabel(contour_lqr, fontsize=10, colors='mediumblue')

plt.title('                    Region of Attraction Specified for V* Computed by NN \n\n                   (fractional-order Lotka-Volterra System) \n', fontsize=12)

# legend for the 3d plot
plt.legend([plt.Rectangle((0,0), 1, 2, color=(94/255, 52/255, 130/255, 0.65), fill=True, linewidth=2),
            plt.Rectangle((0,0), 1, 2, color=(220/255, 80/255, 100/255, 0.65), fill=True, linewidth=2)],
            ['Valid Lyapunov Function V*(X)','Class-Κ Function α(||X||)'], bbox_to_anchor=(0.9, -0.15), loc=0, borderaxespad=0, ncol=1)

plt.xlabel('x1')
plt.ylabel('x2')
plt.savefig('roa_LKS.png', dpi=500, bbox_inches='tight')

plt.show()



# Controlled Dynamics (Compared with The Original One)

In [ ]:
num_steps = 1000

In [ ]:
# actual dynamics
def f_actual(t, X, K):
    x, y = X[0], X[1]

    A = torch.tensor([[a1, 0], [0, -a4]], dtype=torch.float)
    B = torch.tensor([[0], [0]], dtype=torch.float)

    # calculate the linear and controller parts (TENSOR)
    Linear = torch.matmul(A, X.unsqueeze(-1)).squeeze(-1)
    Control = torch.matmul(torch.matmul(B, K), X.unsqueeze(-1)).squeeze(-1)

    # controlled system dynamics
    x_u = Linear[0] - Control[0] - a2*x*y
    y_u = Linear[1] - Control[1] + a3*x*y


    return torch.stack([x_u, y_u])

In [ ]:
f_uncontrolled = fu_solver(f_actual, initial_X_tensor, torch.tensor([[0, 0]], dtype=torch.float), q, t0, Delta_t, num_steps)

In [ ]:
t_interval = np.linspace(t0, t0+((num_steps)*Delta_t), num_steps+1)
lqr_controlled = fu_solver(f_controlled, initial_X_tensor, torch.tensor(K_lqr, dtype=torch.float), q, t0, Delta_t, num_steps)
nn_controlled = fu_solver(f_controlled, initial_X_tensor, torch.tensor(Knn, dtype=torch.float), q, t0, Delta_t, num_steps)

In [ ]:
# Plot the controlled solutions
f_x1, f_x2 = f_uncontrolled[:,0], f_uncontrolled[:,1]
lqr_x1, lqr_x2 = lqr_controlled[:,0], lqr_controlled[:,1]
nn_x1, nn_x2 = nn_controlled[:,0], nn_controlled[:,1]

plt.figure(figsize=(6,5))

# plt.plot(t_interval, lqr_x1, label='LQR_controlled_x1(t)', color='seagreen')
# plt.plot(t_interval, lqr_x2, label='LQR_controlled_x2(t)', color='darkgreen')


plt.plot(t_interval, nn_x1, label='x1(t) (Knn controlled)', color=(50/255, 170/255, 190/255, 0.99), linewidth=1.5)
plt.plot(t_interval, nn_x2, label='x2(t) (Knn controlled)', color=(25/255, 200/255, 130/255, 0.99), linewidth=1.5)

plt.plot(t_interval, f_x1, label='x1(t) (original)', color=(100/255, 10/255, 10/255, 0.75), linewidth=1.5, linestyle='--')
plt.plot(t_interval, f_x2, label='x2(t) (original)', color=(10/255, 10/255, 50/255, 0.75), linewidth=1.5, linestyle='--')
# plt.legend(['original_x(t)', 'original_y(t)'])


plt.legend()
plt.grid(True)
plt.xlabel('Time Steps t')
plt.ylabel('State Variables x')
plt.title('The Original and Controlled Fractional Lotka-Volterra System\n')

plt.tight_layout()
plt.savefig('Controlled_VDP.png', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10,4))

axs[0].plot(t_interval, f_x1, label='original_x(t)', color='mediumblue')
axs[0].plot(t_interval, f_x2, label='original_y(t)', color='royalblue')
axs[0].plot(t_interval, nn_x1, label='controlled_x(t)', color=(30/255, 230/255, 150/255, 0.99))
axs[0].plot(t_interval, nn_x2, label='controlled_y(t)', color=(140/255, 230/255, 30/255, 0.99))
axs[0].legend(['x1(t)', 'x2(t)', 'controlled_x1', 'controlled_x2'], loc='upper left')
axs[0].grid(True)
axs[0].set_xlabel('Time Steps')
axs[0].set_ylabel('State Variables')

axs[1].plot(f_x1, f_x2, color='darkblue')
axs[1].plot(nn_x1, nn_x2, color=(30/255, 170/255, 150/255, 0.99))
axs[1].legend(['Original', 'Controlled'])
axs[1].grid(True)
axs[1].set_xlabel('Original x1(t)')
axs[1].set_ylabel('Original x2(t)')

plt.subplots_adjust(wspace=0.3)

fig.text(0.5, 0.95, 'The Original and Controlled Fractional Lotka-Volterra System\n', horizontalalignment='center', fontsize=14)
plt.savefig('Controlled_VDP.png', dpi=600, bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
V_nn(nn_x1[-1].numpy(), nn_x2[-1].numpy())